[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/stage-3/lab-3.1-deriving-online-softmax.ipynb)

# LAB·3.1 · Deriving online softmax

**Hardware:** any machine. This lab is paper first, numpy second.

Do the paper half before running anything: starting from two-pass softmax, introduce a running (m, l) pair, prove the rescale identity, then fold the value matmul in to get the (m, l, acc) triple. The cells below are only the *check* on your derivation. The one theorem that matters: the combine operation is associative and commutative, which is what makes blocked, streaming, and parallel attention all correct at once.

In [ ]:
import time
import numpy as np
import jax
import jax.numpy as jnp
from jax.experimental import pallas as pl

print(jax.__version__, jax.devices())
ON_TPU = jax.devices()[0].platform == "tpu"
INTERP = not ON_TPU  # interpret mode anywhere; compiled kernels on a real TPU

def check(name, got, want, tol=2e-2):
    err = float(jnp.abs(got.astype(jnp.float32) - want.astype(jnp.float32)).max())
    status = "ok" if err <= tol else "FAIL"
    print(f"{name}: max err {err:.3e} [{status}]")
    assert err <= tol, name


In [ ]:
# the state monoid: combine two partial (m, l, acc) summaries exactly
def combine(a, b):
    m1, l1, acc1 = a
    m2, l2, acc2 = b
    m = jnp.maximum(m1, m2)
    a1, a2 = jnp.exp(m1 - m), jnp.exp(m2 - m)
    return (m, l1 * a1 + l2 * a2, acc1 * a1[..., None] + acc2 * a2[..., None])

def summarize(s_block, v_block):
    m = jnp.max(s_block, axis=-1)
    p = jnp.exp(s_block - m[..., None])
    return (m, jnp.sum(p, axis=-1), p @ v_block)

def streaming_attention(q, k, v, block=64):
    s = q @ k.T
    n = s.shape[-1]
    state = summarize(s[:, :block], v[:block])
    for j in range(block, n, block):
        state = combine(state, summarize(s[:, j:j+block], v[j:j+block]))
    m, l, acc = state
    return acc / l[..., None]

In [ ]:
q = jax.random.normal(jax.random.key(0), (128, 64))
k = jax.random.normal(jax.random.key(1), (512, 64))
v = jax.random.normal(jax.random.key(2), (512, 64))

ref = jax.nn.softmax(q @ k.T, axis=-1) @ v
check("streaming == reference", streaming_attention(q, k, v), ref, tol=1e-5)

## The associativity check

If combine is associative, any bracketing of the blocks gives the same answer, which is precisely what a parallel or distributed reduction needs. Test it the brutal way: random bracketings.

In [ ]:
import random

def reduce_bracketed(states, order):
    states = list(states)
    for idx in order:
        merged = combine(states[idx], states[idx + 1])
        states[idx:idx + 2] = [merged]
    return states[0]

s = q @ k.T
blocks = [summarize(s[:, j:j+64], v[j:j+64]) for j in range(0, 512, 64)]
base = reduce_bracketed(blocks, [0] * (len(blocks) - 1))  # left fold
for trial in range(20):
    order = [random.randrange(n) for n in range(len(blocks) - 1, 0, -1)]
    m, l, acc = reduce_bracketed(blocks, order)
    check(f"bracketing {trial}", acc / l[..., None], base[2] / base[1][..., None], tol=1e-5)
print("combine is associative under 20 random bracketings")

## Keep the notebook

Your paper derivation plus this check is the stage's public artifact seed. If any step of the rescale identity still feels like a trick rather than algebra, redo the paper half; Stage 3's kernels are this identity wearing a schedule.